<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.5_personal_chef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-openai \
    tavily-python

In [1]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
import time
import tavily
import typing
from langchain import chat_models, tools, agents, messages
from langgraph.checkpoint import memory

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

tavily_client = tavily.TavilyClient()
@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:

    """Search the web for information"""

    return tavily_client.search(query=query)

system_prompt = """
You are a personal chef. The user will give you a list of ingredients
they have left over in their house.

Using the web search tool, search the web for recipes that can be made
with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions
to the user, if requested.
"""

agent = agents.create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=memory.InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}
start_time = time.time()
response = agent.invoke(
    {"messages": [messages.HumanMessage(content="""
        I have some leftover chicken and rice. What can I make?
    """)]},
    config
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()


Time taken: 14.24s seconds
================================ Human Message =================================


        I have some leftover chicken and rice. What can I make?
    
================================== Ai Message ==================================
Tool Calls:
  web_search (call_eiajnq7r)
 Call ID: call_eiajnq7r
  Args:
    query: leftover chicken and rice recipes
================================= Tool Message =================================
Name: web_search

{"query": "leftover chicken and rice recipes", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.youtube.com/watch?v=6ioZjlmtKv8", "title": "Leftover Chicken and Rice Recipe For Dinner - YouTube", "content": "This Easy and delicious dinner is made with leftover baked chicken and leftover rice. it's amazing how you can create a quick and easy", "score": 0.9999788, "raw_content": null}, {"url": "https://www.easypeasyfoodie.com/leftover-chicken-egg-fried-rice/", "title": "Leftove